# Phase 3: Validation Framework

This notebook covers:
1. **Performance Evaluation Metrics**: Confusion Matrices, Precision, Recall, F1, ROC-AUC
2. **Cross-Validation Strategies**: Stratified K-Fold vs. Standard K-Fold
3. **Bias-Variance Diagnosis**: Diagnosing Underfitting vs. Overfitting
4. **Hyperparameter Optimization**: Systematic Grid Search CV

In [ ]:
from mlo.data.generator import generate_customer_dataset
from mlo.data.encoder import CategoricalEncoder
from mlo.data.preprocessor import DataPreprocessor
from mlo.evaluation.metrics import calculate_classification_metrics
from mlo.evaluation.validation import perform_stratified_cv
from mlo.evaluation.diagnostics import diagnose_bias_variance
from mlo.evaluation.tuning import tune_hyperparameters
from mlo.utils.visualization import plot_confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import numpy as np

## 1. Cross-Validation & Metric Evaluation

In [ ]:
df = generate_customer_dataset(n_samples=500, random_state=42)
encoder = CategoricalEncoder()
df_encoded = encoder.fit_transform(df)
X = df_encoded[encoder.feature_columns_].values
y = df_encoded['Churn'].values

preproc = DataPreprocessor(scaler_type="standard")
X_scaled = preproc.fit_transform(X)

lr = LogisticRegression(random_state=42)
cv_results = perform_stratified_cv(lr, X_scaled, y, n_splits=5, scoring="f1")
print(f"Stratified 5-Fold F1 Score: {cv_results['mean_score']:.4f} (+/- {cv_results['std_score']:.4f})")

## 2. Confusion Matrix & Bias-Variance Diagnostics

In [ ]:
lr.fit(X_scaled[:400], y[:400])
preds = lr.predict(X_scaled[400:])
probs = lr.predict_proba(X_scaled[400:])[:, 1]
metrics = calculate_classification_metrics(y[400:], preds, probs)

plot_confusion_matrix(np.array(metrics['confusion_matrix']))
diag = diagnose_bias_variance(float(lr.score(X_scaled[:400], y[:400])), float(lr.score(X_scaled[400:], y[400:])))
print("Diagnosis:", diag['diagnosis'])
print("Remedy:", diag['recommendation'])

## 3. Systematic Grid Search CV Tuning

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
param_grid = {'max_depth': [2, 3, 5, 8], 'criterion': ['gini', 'entropy'], 'min_samples_split': [2, 5, 10]}
best_tune = tune_hyperparameters(dt, param_grid, X_scaled, y, method="grid", cv=5, scoring="f1")
print("Optimal Hyperparameters:", best_tune['best_params'])
print("Best F1 Score:", round(best_tune['best_score'], 4))